# Prompt Strategy Test Notebook

Notebook này dùng để test output của prompt theo từng strategy: `zero-shot`, `few-shot`, `cot`.

Mục tiêu:
1. So sánh phản hồi giữa các strategy trên cùng câu.
2. Kiểm tra format tuple có đúng chuẩn hay không.
3. Kiểm tra span constraint: cụm từ dự đoán phải xuất hiện trong câu gốc.

In [ ]:
# Cell 2 · Setup imports and paths
import os
import re
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if (ROOT / 'llm_eval').exists():
    PROJECT_ROOT = ROOT
elif ROOT.name == 'llm_eval' and (ROOT / 'prompts.py').exists():
    PROJECT_ROOT = ROOT.parent
else:
    PROJECT_ROOT = ROOT

LLM_EVAL_DIR = PROJECT_ROOT / 'llm_eval'
sys.path.insert(0, str(LLM_EVAL_DIR))

from prompts import build_messages
from client import OpenAICompatibleClient, HuggingFaceLocalClient

print('Project root:', PROJECT_ROOT)
print('llm_eval dir:', LLM_EVAL_DIR)

Project root: /home/haiyan/msc-project
llm_eval dir: /home/haiyan/msc-project/llm_eval


In [2]:
# Cell 3 · Configuration
PROVIDER = 'openrouter'          # openrouter | hf-local
MODEL = 'openai/gpt-4o-mini'    # or HF model id
BASE_URL = 'https://openrouter.ai/api/v1'
API_KEY_ENV = 'OPENROUTER_API_KEY'

HF_DTYPE = 'auto'                # auto | float16 | bfloat16
HF_LOAD_IN_4BIT = False

TEMPERATURE = 0.0
MAX_OUTPUT_TOKENS = 256

DATASET = 'camera-coqe'            # vcom-data | camera-coqe | t5-camera-coqe-data
LANGUAGE = 'en'                  # vi | en | auto
OUTPUT_FORMAT = 'json'           # json | textual

STRATEGIES = ['zero-shot', 'few-shot', 'cot']

TEST_SENTENCES = [
    'Bên cạnh đó, iPhone 14 được nâng cấp bộ nhớ lên đến 6GB RAM cao hơn iPhone 13 đến 2GB RAM, cho khả năng đa nhiệm tốt hơn.',
    'Tương tự, thì ống kính góc rộng không có quá nhiều sự khác biệt so với ống kính chính.',
    'Bạn có thể selfie và sử dụng ở bể bơi mà không hề sợ bị hỏng máy.',
]

print('Configured provider:', PROVIDER)
print('Configured model   :', MODEL)
print('Output format      :', OUTPUT_FORMAT)

Configured provider: openrouter
Configured model   : openai/gpt-4o-mini
Output format      : json


In [3]:
# Preview prompts only (no model call)
# This helps inspect exactly what the current prompt templates look like.

sentence_preview = TEST_SENTENCES[0] if TEST_SENTENCES else ''
print('Preview sentence:', sentence_preview)

for strategy in STRATEGIES:
    messages = build_messages(
        sentence=sentence_preview,
        language=LANGUAGE,
        dataset=DATASET,
        strategy=strategy,
        output_format=OUTPUT_FORMAT,
    )

    print('=' * 120)
    print(f'STRATEGY: {strategy}')
    print('-' * 120)
    print('[SYSTEM]')
    print(messages[0]['content'])
    print('-' * 120)
    print('[USER]')
    print(messages[1]['content'])
    print()

Preview sentence: Bên cạnh đó, iPhone 14 được nâng cấp bộ nhớ lên đến 6GB RAM cao hơn iPhone 13 đến 2GB RAM, cho khả năng đa nhiệm tốt hơn.
STRATEGY: zero-shot
------------------------------------------------------------------------------------------------------------------------
[SYSTEM]
You are an information extraction model for comparative opinion mining. 
Given one sentence, your task is to analyze and extract all comparative opinion quintuples if the sentence contains any comparative meaning. You must follow these rules:
- Analyze and extract 5 components of each comparison: comparative subject, comparative object, comparative aspect, comparative predicate, and comparative label.
- A sentence may contain one or multiple relations; extract all of them.
- subject, object, aspect, predicate must be verbatim spans from the original sentence. 
- If a component is implicit but can be clearly inferred from the context in the sentence, use the specific words for that entity that appeared

In [7]:
# Cell 4 · Build client
if PROVIDER == 'openrouter':
    client = OpenAICompatibleClient(
        model=MODEL,
        base_url=BASE_URL,
        api_key_env=API_KEY_ENV,
        temperature=TEMPERATURE,
        max_output_tokens=MAX_OUTPUT_TOKENS,
    )
elif PROVIDER == 'hf-local':
    client = HuggingFaceLocalClient(
        model=MODEL,
        temperature=TEMPERATURE,
        max_output_tokens=MAX_OUTPUT_TOKENS,
        dtype=HF_DTYPE,
        load_in_4bit=HF_LOAD_IN_4BIT,
    )
else:
    raise ValueError(f'Unsupported PROVIDER: {PROVIDER}')

print('Client initialized.')

NameError: name 'OpenAICompatibleClient' is not defined

In [ ]:
# Cell 5 · Format validators
import json

TUPLE_RE = re.compile(
    r'\[S\]\s*(.*?)\s*\[O\]\s*(.*?)\s*\[A\]\s*(.*?)\s*\[P\]\s*(.*?)\s*\[L\]\s*(.*?)(?=\)|\n|;|$)',
    re.DOTALL,
)

EMPTY_TUPLE = '([S] [UNK] [O] [UNK] [A] [UNK] [P] [UNK] [L] [UNK])'

def parse_tuples_text(output_text: str):
    tuples = []
    for part in (output_text or '').split(';'):
        m = TUPLE_RE.search(part.strip().strip('()'))
        if m:
            s, o, a, p, l = [x.strip() for x in m.groups()]
            tuples.append({'S': s, 'O': o, 'A': a, 'P': p, 'L': l})
    return tuples

def _extract_json_text(output_text: str):
    text = (output_text or '').strip()
    if text.startswith('```'):
        m = re.search(r'```(?:json)?\s*(.*?)\s*```', text, flags=re.DOTALL | re.IGNORECASE)
        if m:
            text = m.group(1).strip()

    starts = [i for i in [text.find('{'), text.find('[')] if i >= 0]
    if starts:
        st = min(starts)
        en = max(text.rfind('}'), text.rfind(']'))
        if en > st:
            text = text[st:en+1]
    return text

def parse_tuples_json(output_text: str):
    text = _extract_json_text(output_text)
    data = json.loads(text)
    if isinstance(data, dict):
        if isinstance(data.get('comparisons'), list):
            items = data['comparisons']
        elif isinstance(data.get('quintuples'), list):
            items = data['quintuples']
        elif any(k in data for k in ('S', 'O', 'A', 'P', 'L', 'subject', 'object', 'aspect', 'predicate', 'label')):
            items = [data]
        else:
            items = []
    elif isinstance(data, list):
        items = data
    else:
        items = []

    tuples = []
    for item in items:
        if not isinstance(item, dict):
            continue

        label_val = item.get('L', item.get('label', '[UNK]'))
        if isinstance(label_val, str) and label_val.strip().upper() == 'DIFF':
            label_val = 'DIF'

        entities_val = item.get('entities', [])
        if isinstance(entities_val, list):
            ent_s = entities_val[0] if len(entities_val) > 0 else '[UNK]'
            ent_o = entities_val[1] if len(entities_val) > 1 else '[UNK]'
        else:
            ent_s = '[UNK]'
            ent_o = '[UNK]'

        tuples.append({
            'S': str(item.get('S', item.get('subject', ent_s))).strip() or '[UNK]',
            'O': str(item.get('O', item.get('object', ent_o))).strip() or '[UNK]',
            'A': str(item.get('A', item.get('aspect', '[UNK]'))).strip() or '[UNK]',
            'P': str(item.get('P', item.get('predicate', '[UNK]'))).strip() or '[UNK]',
            'L': str(label_val).strip() or '[UNK]',
        })
    return tuples

def _is_valid_empty_json_output(output_text: str):
    try:
        data = json.loads(_extract_json_text(output_text))
    except Exception:
        return False

    if isinstance(data, dict):
        if isinstance(data.get('comparisons'), list) and len(data['comparisons']) == 0:
            return True
        if isinstance(data.get('quintuples'), list) and len(data['quintuples']) == 0:
            return True
    return False

def parse_tuples(output_text: str):
    text = (output_text or '').strip()
    if OUTPUT_FORMAT == 'json':
        try:
            return parse_tuples_json(text)
        except Exception:
            return []
    return parse_tuples_text(text)

def validate_format(output_text: str):
    text = (output_text or '').strip()
    tuples = parse_tuples(text)
    if OUTPUT_FORMAT == 'json':
        ok_structure = (len(tuples) > 0) or _is_valid_empty_json_output(text)
    else:
        ok_structure = (text == EMPTY_TUPLE) or (len(tuples) > 0)
    return {
        'ok_structure': ok_structure,
        'tuple_count': len(tuples),
        'parsed_tuples': tuples,
    }

def validate_span_constraint(sentence: str, parsed_tuples):
    sent_low = sentence.lower()
    violations = []
    for i, t in enumerate(parsed_tuples, start=1):
        for slot in ('S', 'O', 'A', 'P'):
            val = (t.get(slot) or '').strip()
            if not val or val == '[UNK]':
                continue
            if val.lower() not in sent_low:
                violations.append({'tuple_idx': i, 'slot': slot, 'value': val})
    return {
        'ok_span_constraint': len(violations) == 0,
        'violations': violations,
    }

In [ ]:
# Cell 6 · Run prompt tests by strategy
import json
from datetime import datetime

results = []

# Raw log file: one JSON object per (sentence, strategy)
out_dir = LLM_EVAL_DIR / 'results' / 'prompt_test'
out_dir.mkdir(parents=True, exist_ok=True)
run_ts = datetime.now().strftime('%Y%m%d_%H%M%S')
raw_log_file = out_dir / f'raw_outputs__{PROVIDER}__{run_ts}.jsonl'

with open(raw_log_file, 'w', encoding='utf-8') as log_fp:
    for sent_idx, sentence in enumerate(TEST_SENTENCES, start=1):
        print('= ' * 50)
        print(f'Sentence {sent_idx}: {sentence}')

        for strategy in STRATEGIES:
            messages = build_messages(
                sentence=sentence,
                language=LANGUAGE,
                dataset=DATASET,
                strategy=strategy,
                output_format=OUTPUT_FORMAT,
            )

            output = client.generate(messages)
            fmt = validate_format(output)
            span = validate_span_constraint(sentence, fmt['parsed_tuples'])

            record = {
                'sentence_idx': sent_idx,
                'strategy': strategy,
                'output': output,
                'ok_structure': fmt['ok_structure'],
                'tuple_count': fmt['tuple_count'],
                'ok_span_constraint': span['ok_span_constraint'],
                'violations': span['violations'],
            }
            results.append(record)

            # Persist raw trace for debugging and reproducibility.
            raw_row = {
                'timestamp': run_ts,
                'provider': PROVIDER,
                'model': MODEL,
                'dataset': DATASET,
                'language': LANGUAGE,
                'output_format': OUTPUT_FORMAT,
                'sentence_idx': sent_idx,
                'sentence': sentence,
                'strategy': strategy,
                'messages': messages,
                'output': output,
                'validation': {
                    'ok_structure': fmt['ok_structure'],
                    'ok_span_constraint': span['ok_span_constraint'],
                    'tuple_count': fmt['tuple_count'],
                    'violations': span['violations'],
                },
            }
            log_fp.write(json.dumps(raw_row, ensure_ascii=False) + '\n')

            status = 'PASS' if (fmt['ok_structure'] and span['ok_span_constraint']) else 'FAIL'
            print(f'[{strategy}] => {status}')
            print('Output:', output)
            if not span['ok_span_constraint']:
                print('Span violations:', span['violations'])
            print('-' * 100)

print('Raw output log saved to:', raw_log_file)

In [ ]:
# Cell 7 · Summary table
try:
    import pandas as pd
    df = pd.DataFrame(results)
    show_cols = ['sentence_idx', 'strategy', 'ok_structure', 'ok_span_constraint', 'tuple_count']
    print(df[show_cols].to_string(index=False))
except ImportError:
    for r in results:
        print({k: r[k] for k in ['sentence_idx', 'strategy', 'ok_structure', 'ok_span_constraint', 'tuple_count']})

In [ ]:
# Cell 8 · Save test report (optional)
import json
from datetime import datetime

out_dir = LLM_EVAL_DIR / 'results' / 'prompt_test'
out_dir.mkdir(parents=True, exist_ok=True)

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
out_file = out_dir / f'prompt_test__{PROVIDER}__{ts}.json'

payload = {
    'provider': PROVIDER,
    'model': MODEL,
    'dataset': DATASET,
    'language': LANGUAGE,
    'strategies': STRATEGIES,
    'results': results,
}

with open(out_file, 'w', encoding='utf-8') as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print('Saved report to:', out_file)